In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
mohamedmaher5_vehicle_classification_path = kagglehub.dataset_download('mohamedmaher5/vehicle-classification')

print('Data source import complete.')


100%|██████████| 827M/827M [00:14<00:00, 58.7MB/s]

Extracting files...


Data source import complete.


## Step 1 : Downlaod Dataset

In [2]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("mohamedmaher5/vehicle-classification")
print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/mohamedmaher5/vehicle-classification/versions/1


In [7]:
!pip install split-folders

## Step 2 : Import Dependencies

In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import numpy as np
import gradio as gr
from PIL import Image
import random
import splitfolders
import os
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, BatchNormalization, Dropout
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt


## Step 3 : Data Spliting (Train, Val, Test)

In [9]:
# Set a seed for reproducibility
random.seed(42)

# Define the source directory and destination directory
source_dir = "/root/.cache/kagglehub/datasets/mohamedmaher5/vehicle-classification/versions/1"
dest_dir = "data"  # This folder will be created in the current working directory

# Check if source_dir contains a single directory (e.g., 'Vehicles') and update if needed
subdirs = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]
if len(subdirs) == 1:
    source_dir = os.path.join(source_dir, subdirs[0])
    print("Updated source directory to:", source_dir)




# Split the folder into train, validation, and test sets.
# The ratio corresponds to (train, validation, test).
splitfolders.ratio(source_dir, output=dest_dir, seed=42, ratio=(0.7, 0.2, 0.1))

print("Dataset splitting completed.")

Updated source directory to: /root/.cache/kagglehub/datasets/mohamedmaher5/vehicle-classification/versions/1/Vehicles


Copying files: 5590 files [00:07, 732.73 files/s]

Dataset splitting completed.


## Step 4 : Build Data Generators

In [10]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define image size and batch size
img_width, img_height = 150, 150
batch_size = 32

# Training generator with data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

# Validation and test generators: only rescaling
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)


dest_dir = "data"

# Create generators from the split folders
train_generator = train_datagen.flow_from_directory(
    os.path.join(dest_dir, 'train'),
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)


validation_generator = val_datagen.flow_from_directory(
    os.path.join(dest_dir, 'val'),
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    os.path.join(dest_dir, 'test'),
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False  # Keep order for evaluation
)


Found 3911 images belonging to 7 classes.
Found 1118 images belonging to 7 classes.
Found 558 images belonging to 7 classes.


## Step 5 : Define Model Architecture

### Trianing Model on One GPU

In [ ]:
# -------------------------------
# Step 3: Build and Train the CNN Model
# -------------------------------
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Determine the number of classes from the training generator
num_classes = len(train_generator.class_indices)

# Build a simple CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_width, img_height, 3)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

print(model.summary())

# Compile the model
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])


# Train the model using training and validation data
epochs = 5
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator
)


# save the model and label binarizer to disk
print("[INFO] serializing network ...")
model.save("cnn_model_{}.h5".format(epochs))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 36, 36, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 34, 34, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 17, 17, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       4,735,104 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 7)                   │             903 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,829,255 (18.42 MB)

 Trainable params: 4,829,255 (18.42 MB)

...

None


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
 34/123 ━━━━━━━━━━━━━━━━━━━━ 2:47 2s/step - accuracy: 0.1816 - loss: 1.9624

/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


 44/123 ━━━━━━━━━━━━━━━━━━━━ 2:36 2s/step - accuracy: 0.1821 - loss: 1.9559

### Traning model on Multiple GPU

In [ ]:


# Update input dimensions to match your generator's output (150x150 images)
height, width, channels = 150, 150, 3
num_classes = 7  # Update this according to your dataset

# Create a MirroredStrategy instance for multi-GPU training
strategy = tf.distribute.MirroredStrategy()
print("Number of devices: {}".format(strategy.num_replicas_in_sync))

with strategy.scope():
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(height, width, channels)),
        BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        # Block 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        # Block 4: Additional convolutional layers for increased depth
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        # Flatten and fully connected layers
        Flatten(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

# Train the model using your data generators
epochs = 5
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator
)

# Save the trained model to disk
print("[INFO] serializing network ...")
model.save("cnn_model_{}.h5".format(epochs))


## Step 6: Model Accuracy and Loss Plot

In [ ]:
N = np.arange(0, epochs)
plt.style.use("ggplot")
plt.figure(figsize = [10,8])
plt.plot(N, history.history["loss"], label="train_loss")
plt.plot(N, history.history["val_loss"], label="val_loss")
plt.title("CNN: Training & Validation Loss")
plt.xlabel("Epoch #", weight="bold")
plt.ylabel("Loss", weight="bold")
plt.legend()

In [ ]:

N = np.arange(0, epochs )
plt.style.use("ggplot")
plt.figure(figsize = [10,8])
plt.plot(N, history.history["accuracy"], label="train_acc")
plt.plot(N, history.history["val_accuracy"], label="val_acc")
plt.title("CNN: Training and Validation Accuracy")
plt.xlabel("Epoch #", weight="bold")
plt.ylabel("Accuracy", weight="bold")
plt.legend()
plt.show()

## Step 7 : Model Evaluation on Test Data

In [ ]:

# Reset the test generator and get predictions
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# Get the class labels
class_labels = list(test_generator.class_indices.keys())

# Generate and print the classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_labels))

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(class_labels))
plt.xticks(tick_marks, class_labels, rotation=45)
plt.yticks(tick_marks, class_labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

# Annotate each cell with the count
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.show()

## Step 8 : Model Deployment Using Gradio

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image
import tensorflow as tf

# Load the trained model
model = tf.keras.models.load_model("cnn_model_5.h5")

# Set the image dimensions (should match training settings)
img_width, img_height = 150, 150

# Define class labels mapping
class_labels = {
    0: "Auto Rickshaws",
    1: "Bikes",
    2: "Cars",
    3: "Motorcycles",
    4: "Planes",
    5: "Ships",
    6: "Trains"
}

def predict_vehicle(input_image: Image.Image) -> str:
    # Resize and preprocess the image
    img = input_image.resize((img_width, img_height))
    img_array = np.array(img) / 255.0  # Normalize the image
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Get model predictions
    predictions = model.predict(img_array)
    predicted_index = int(np.argmax(predictions, axis=1)[0])

    # Return the predicted label
    return class_labels[predicted_index]

# Create a Gradio interface using explicit component definitions
interface = gr.Interface(
    fn=predict_vehicle,
    inputs=gr.Image(type="pil", label="Upload a Vehicle Image"),
    outputs=gr.Textbox(label="Prediction"),
    title="Vehicle Classification",
    description="Upload an image of a vehicle and the model will classify it into one of 7 categories."
)

# Launch the Gradio app (using share=True if needed for external access)
interface.launch(share=True)


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'cnn_model_5.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)